# Detecting brain tumors in MRI with YOLO11n

Open this notebook in Google Colab with a T4 GPU and select **Runtime → Run all**. The default mode trains a 50-epoch baseline and two controlled experiments, compares their best checkpoints on validation, and evaluates the selected checkpoint on the held-out test split.

Choose `saved` in the settings cell to run evaluation and prediction with the project's saved weights. Colab downloads the split manifest and saved results from the GitHub repository. You can also supply the prepared `subset_v1.zip` in `/content`. The notebook downloads the original [Kaggle dataset](https://www.kaggle.com/datasets/ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes) when it needs to rebuild the subset.

Each execution writes its own directory and ZIP archive under `/content/brain_tumor/runs`. Set `save_to_drive = True` to save them in your Google Drive, or download the ZIP from the Colab Files panel before ending the runtime.

## 1. Install dependencies

In [ ]:
%pip install -q ultralytics==8.4.190 albumentations==2.0.8 pandas==2.2.3

In [ ]:
import hashlib
import json
import logging
import os
import shutil
import tempfile
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import display
from ultralytics import YOLO
from ultralytics.utils import LOGGER
from ultralytics.utils.metrics import smooth

in_colab = Path("/content").is_dir()
mode = os.environ.get("BT_MODE", "train")  # "train" or "saved"
save_to_drive = False
smoke_test = os.environ.get("BT_SMOKE_TEST") == "1"
assert mode in {"train", "saved"}
assert ultralytics.__version__ == "8.4.190", "Restart the kernel after installing dependencies."

repo_dir = Path(os.environ.get("BT_REPO_DIR", Path.cwd())).resolve()
work_dir = Path(os.environ.get("BT_WORK_DIR", "/content/brain_tumor" if in_colab else repo_dir / "data" / "colab")).resolve()
work_dir.mkdir(parents=True, exist_ok=True)
output_root = work_dir / "runs"
if save_to_drive:
    from google.colab import drive

    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/brain_tumor/colab_runs")

session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = output_root / f"{mode}_{session_id}"
output_dir.mkdir(parents=True)
device = 0 if torch.cuda.is_available() else "cpu"
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Mode: {mode} | Device: {device_name} | Output: {output_dir}")

## 2. Prepare the fixed data split

The project uses 1,350 images across train (945), validation (202), and test (203). The split manifest records the source file, class, and group for each image. Labels are normalized YOLO boxes. Glioma, Meningioma, and Pituitary are the three detection classes. Images in the No Tumor folder have empty labels and serve as background examples.

In [ ]:
REPO_RAW = "https://raw.githubusercontent.com/Kanaagaat/brain-tumor-object-detection/main"
KAGGLE_ZIP = ("https://www.kaggle.com/api/v1/datasets/download/"
              "ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes?datasetVersionNumber=1")


def download(url, target):
    """Keep incomplete downloads out of the dataset and checkpoint caches."""
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".part")
    urlretrieve(url, partial)
    partial.replace(target)
    return target


def repo_file(relative):
    local = repo_dir / relative
    if local.is_file():
        return local
    cached = work_dir / "repository" / relative
    return cached if cached.is_file() else download(f"{REPO_RAW}/{relative}", cached)


def rebuild_subset(target):
    manifest_path = repo_file("reports/split_v1_manifest.csv")
    config_path = repo_file("reports/split_v1_config.json")
    source_zip = Path(os.environ.get("BT_KAGGLE_ZIP", work_dir / "kaggle_v1.zip"))
    if not source_zip.is_file():
        print("Downloading the original Kaggle dataset (version 1).")
        download(KAGGLE_ZIP, source_zip)

    manifest = pd.read_csv(manifest_path, keep_default_na=False)
    class_remap = {0: 0, 1: 1, 3: 2}
    for split in ("train", "val", "test"):
        (target / "images" / split).mkdir(parents=True)
        (target / "labels" / split).mkdir(parents=True)
    with zipfile.ZipFile(source_zip) as archive:
        first = manifest.iloc[0]["source_image"]
        first_member = next(name for name in archive.namelist() if name.endswith(first))
        prefix = first_member.removesuffix(first)
        for row in manifest.itertuples(index=False):
            source = Path(row.source_image)
            image = archive.read(prefix + source.as_posix())
            if hashlib.md5(image).hexdigest() != row.md5:
                raise ValueError(f"Image differs from the fixed split: {source}")
            (target / "images" / row.split / row.file).write_bytes(image)
            labels = []
            if row.folder != "No Tumor":
                source_label = source.parent.parent / "labels" / (source.stem + ".txt")
                for line in archive.read(prefix + source_label.as_posix()).decode().splitlines():
                    if line.strip():
                        class_id, *coords = line.split()
                        if int(class_id) in class_remap:
                            labels.append(f"{class_remap[int(class_id)]} " + " ".join(str(float(x)) for x in coords))
            if len(labels) != row.n_boxes_new:
                raise ValueError(f"Unexpected number of annotations for {source}: {len(labels)}")
            (target / "labels" / row.split / f"{Path(row.file).stem}.txt").write_text(
                "".join(f"{line}\n" for line in labels)
            )
    shutil.copy2(manifest_path, target / "split_manifest.csv")
    shutil.copy2(config_path, target / "split_config.json")


def validate_subset(target):
    config = json.loads((target / "split_config.json").read_text())
    manifest = pd.read_csv(target / "split_manifest.csv", keep_default_na=False)
    expected = {"train": 945, "val": 202, "test": 203}
    if config["images_per_split"] != expected or manifest["split"].value_counts().to_dict() != expected:
        raise ValueError("The dataset does not match the project's fixed split sizes.")
    if manifest["md5"].duplicated().any() or manifest.groupby("group")["split"].nunique().max() != 1:
        raise ValueError("The split contains duplicate images or groups across splits.")
    for row in manifest.itertuples(index=False):
        image = target / "images" / row.split / row.file
        label = target / "labels" / row.split / f"{Path(row.file).stem}.txt"
        if hashlib.md5(image.read_bytes()).hexdigest() != row.md5:
            raise ValueError(f"Unexpected image contents: {image}")
        if len(label.read_text().splitlines()) != row.n_boxes_new:
            raise ValueError(f"Unexpected number of annotations: {label}")
    return config, manifest


DATA_DIR = work_dir / "subset_v1"
if not DATA_DIR.exists():
    prepared_archives = [
        Path(os.environ.get("BT_DATA_ZIP", "/content/subset_v1.zip")),
        repo_dir / "data" / "subset_v1.zip",
    ]
    prepared_zip = next((path for path in prepared_archives if path.is_file()), None)
    with tempfile.TemporaryDirectory(dir=work_dir) as staging_dir:
        staged = Path(staging_dir) / DATA_DIR.name
        if prepared_zip:
            print(f"Extracting the prepared dataset: {prepared_zip}")
            with zipfile.ZipFile(prepared_zip) as archive:
                archive.extractall(staging_dir)
        else:
            rebuild_subset(staged)
        split_config, manifest = validate_subset(staged)
        staged.rename(DATA_DIR)
else:
    split_config, manifest = validate_subset(DATA_DIR)

class_names = split_config["class_names"]
DATA_YAML = work_dir / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_DIR), "train": "images/train", "val": "images/val", "test": "images/test",
    "names": dict(enumerate(class_names)),
}))
display(pd.DataFrame(split_config["images_per_split_and_folder"]).rename_axis("Images"))
display(pd.DataFrame(split_config["objects_per_split_and_class"]).rename_axis("Boxes"))

## 3. Train three YOLO11n configurations

YOLO11n fit the Colab T4 budget and was fine-tuned from COCO weights. The baseline uses SGD, 50 epochs, an image size of 512, and seed 42. Both experiments start from the same `yolo11n.pt` and use the same split.

- **100 epochs.** The rationale for the longer schedule was that extra training could improve localization. This rationale was documented after the historical run. The epoch budget also controls the learning-rate schedule and when mosaic is disabled.
- **Lower learning rate.** The hypothesis was that reducing `lr0` from 0.01 to 0.005 would improve validation mAP by adapting the pretrained weights more gradually. The comparison uses 50 epochs for both runs.

In [ ]:
BASELINE_NAME = "baseline_yolo11n_512"
CHANGES = {
    BASELINE_NAME: {},
    "exp_epochs100": {"epochs": 100},
    "exp_lr00.005": {"lr0": 0.005},
}
base_settings = {
    "imgsz": 512, "epochs": 50, "batch": 16,
    "optimizer": "SGD", "lr0": 0.01, "lrf": 0.01,
    "momentum": 0.937, "weight_decay": 0.0005, "warmup_epochs": 3.0,
    "cos_lr": False, "patience": 50, "seed": 42, "deterministic": True,
    "workers": 2, "plots": True,
}
validation_settings = {"split": "val", "conf": 0.001, "iou": 0.7}

if smoke_test:
    base_settings.update(imgsz=256, epochs=1, batch=8, fraction=0.05, workers=0)
    CHANGES["exp_epochs100"]["epochs"] = 2


def metrics(result):
    box = result.box
    return {"precision": box.mp, "recall": box.mr, "mAP50": box.map50, "mAP50-95": box.map}


def pretrained_checkpoint():
    override = os.environ.get("BT_INITIAL_WEIGHTS")
    if override:
        return Path(override).resolve(strict=True)
    checkpoint = work_dir / "yolo11n.pt"
    if not checkpoint.is_file():
        url = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.pt"
        download(url, checkpoint)
    return checkpoint


def train_run(name, changes):
    settings = {**base_settings, **changes}
    model = YOLO(initial_weights)
    log_path = output_dir / f"{name}_train_log.txt"
    handler = logging.FileHandler(log_path, mode="w")
    LOGGER.addHandler(handler)
    start = time.perf_counter()
    try:
        model.train(data=str(DATA_YAML), device=device, project=str(output_dir), name=name,
                    exist_ok=False, **settings)
    finally:
        LOGGER.removeHandler(handler)
        handler.close()
    train_minutes = (time.perf_counter() - start) / 60
    run_dir = Path(model.trainer.save_dir)
    log_path.rename(run_dir / "train_log.txt")

    validation = YOLO(run_dir / "weights" / "best.pt").val(
        data=str(DATA_YAML), imgsz=settings["imgsz"], batch=settings["batch"], device=device,
        plots=True, project=str(run_dir), name="val_best", **validation_settings,
    )
    history = pd.read_csv(run_dir / "results.csv")
    history.columns = history.columns.str.strip()
    best_epoch = int(history.loc[history["metrics/mAP50-95(B)"].idxmax(), "epoch"])
    info = {
        "run_name": name,
        "initial_weights": initial_weights.name,
        "initial_weights_sha256": initial_weights_sha256,
        "settings": settings,
        "changed_setting": changes,
        "versions": {"ultralytics": ultralytics.__version__, "torch": torch.__version__,
                     "pandas": pd.__version__, "device": device_name},
        "dataset": {"version": split_config["dataset_version"], "split_seed": split_config["random_seed"]},
        "training_time_minutes": train_minutes,
        "best_epoch": best_epoch,
        "best_checkpoint": "weights/best.pt",
        "validation_settings": validation_settings,
        "validation_metrics_best": metrics(validation),
    }
    (run_dir / "run_info.json").write_text(json.dumps(info, indent=2))
    return info


if mode == "train":
    initial_weights = pretrained_checkpoint()
    initial_weights_sha256 = hashlib.sha256(initial_weights.read_bytes()).hexdigest()
    run_reports = {name: train_run(name, changes) for name, changes in CHANGES.items()}
else:
    run_reports = {
        name: json.loads(repo_file(f"runs/{name}/run_info.json").read_text()) for name in CHANGES
    }

## 4. Compare on validation and select one checkpoint

The historical results below were recorded using the same validation settings. The lower learning rate yielded the highest mAP@0.50. The 100-epoch run had the highest mAP averaged over IoU thresholds from 0.50 through 0.95, the criterion used to select the final checkpoint. It took 19.98 minutes versus 10.70 for baseline. The precision and recall values are measured at the confidence giving the highest mean F1 for each model.

| Run | Epochs | Initial LR | Precision | Recall | mAP@0.50 | mAP@0.50–0.95 | Train min |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | 50 | 0.010 | 0.870 | 0.901 | 0.934 | 0.692 | 10.70 |
| Longer schedule | 100 | 0.010 | 0.907 | 0.889 | 0.936 | **0.707** | 19.98 |
| Lower LR | 50 | 0.005 | 0.917 | 0.910 | **0.949** | 0.697 | 11.23 |

The lower learning rate result supports the hypothesis within this run. Its gain at IoU 0.50 (+0.015) was larger than its gain across IoU 0.50–0.95 (+0.005). For Glioma, mAP@0.50 improved from 0.933 to 0.945 and mAP@0.50–0.95 fell from 0.685 to 0.664. The overall gain varied by class and overlap threshold. The longer run traded additional training time and lower recall at its selected operating point for higher mAP@0.50–0.95. These comparisons use one seed per setting.

The next cell applies the same validation criterion to the runs loaded or trained in this session.

In [ ]:
comparison = pd.DataFrame.from_dict({
    name: {
        "epochs": report["settings"]["epochs"],
        "lr0": report["settings"]["lr0"],
        **report["validation_metrics_best"],
        "best epoch": report["best_epoch"],
        "train min": report["training_time_minutes"],
    }
    for name, report in run_reports.items()
}, orient="index")
comparison.to_csv(output_dir / "validation_comparison.csv")
display(comparison.round(3))

selected_run = comparison["mAP50-95"].idxmax()
selected_info = run_reports[selected_run]
checkpoint_relative = Path(selected_run) / "weights" / "best.pt"
checkpoint = output_dir / checkpoint_relative
if mode == "saved":
    saved_weights = repo_file(Path("runs") / checkpoint_relative)
    checkpoint.parent.mkdir(parents=True)
    shutil.copy2(saved_weights, checkpoint)
print(f"Selected by validation mAP@0.50:0.95: {selected_run}")
print(f"Checkpoint: {checkpoint}")

## 5. Evaluate and show predictions on test

Precision is the share of detections that match an annotated box and class. Recall is the share of annotated objects found. mAP@0.50 averages the area under the precision–recall curve across classes, using IoU ≥ 0.50 to match a prediction to a reference box. Ultralytics reports precision and recall at the confidence that maximizes mean F1, while mAP uses detections from a low confidence floor of 0.001. The display uses confidence 0.25. NMS IoU is 0.7 for both.

In [ ]:
test_dir = output_dir / "test_eval"
test_dir.mkdir()
test_settings = {
    **validation_settings, "split": "test",
    "imgsz": selected_info["settings"]["imgsz"], "batch": selected_info["settings"]["batch"],
}
model = YOLO(checkpoint)
result = model.val(data=str(DATA_YAML), device=device, plots=True,
                   project=str(test_dir), name="ultralytics", **test_settings)
confidence_at_max_f1 = float(result.box.px[smooth(result.box.f1_curve.mean(0), 0.1).argmax()])
test_scores = metrics(result)
(test_dir / "test_metrics.json").write_text(json.dumps({
    "run": selected_run,
    "selected_by": "validation mAP@0.50:0.95",
    "settings": test_settings,
    "precision_recall_at_conf": confidence_at_max_f1,
    "metrics": test_scores,
}, indent=2))
display(pd.DataFrame([test_scores], index=[selected_run]).round(3))
print(f"Precision and recall operating point: confidence {confidence_at_max_f1:.3f}")

examples = manifest[manifest["split"] == "test"].groupby("folder").sample(n=2, random_state=42)
image_paths = [str(DATA_DIR / "images" / "test" / name) for name in examples["file"]]
predictions = model.predict(image_paths, imgsz=test_settings["imgsz"],
                            conf=0.25, iou=0.7, device=device, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), layout="constrained")
for ax, example, pred in zip(axes.flat, examples.itertuples(), predictions):
    ax.imshow(pred.plot()[..., ::-1])
    ax.set_title(f"{example.folder} · {len(pred.boxes)} predictions")
    ax.axis("off")
fig.suptitle("Held-out test examples · class labels and confidence scores")
fig.savefig(test_dir / "test_predictions.png", dpi=120)
plt.show()

In [ ]:
archive_path = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir.parent, base_dir=output_dir.name)
print(f"Best weights: {checkpoint}")
print(f"Download this archive from the Files panel: {archive_path}")